# ODA-LAB extract — named packs only

**Runtime → Run all.** Allow Drive once.

Cap 50 MB. Skips dest if it exists. Will not touch grok-five 416 MB.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, zipfile, time
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive')
print('mounted', ROOT.exists())


In [ ]:
CAP = 50 * 1024 * 1024
PACKS = [
  'grokbot/from-olivia/2026-09-10_SUNSET_this_conversation_Olivia_pixels_artifacts/2026-09-10_SUNSET_TEXT_PACKAGES.zip',
  'grokbot/from-olivia/sunset_2026-09-10_pixels-and-artifacts/sunset_pixels_other_2026-09-10.zip',
  'grokbot/from-olivia/sunset_full_archive_2026-09-10.zip',
]
shelf = None
for cand in ROOT.rglob('12_ODA-LAB-NOTEBOOKS'):
    if cand.is_dir():
        shelf = cand
        break
if shelf is None:
    shelf = ROOT / '12_ODA-LAB-NOTEBOOKS_LOCAL'
    shelf.mkdir(exist_ok=True)
dest_root = shelf / 'extracts' / time.strftime('%Y%m%d-%H%M')
dest_root.mkdir(parents=True, exist_ok=True)
lines = ['# ODA LAB extract receipt', 'dest=' + str(dest_root), '']
for rel in PACKS:
    src = ROOT / rel
    print('---', src)
    if not src.is_file():
        print('MISSING')
        lines.append('- MISSING ' + rel)
        continue
    sz = src.stat().st_size
    print('size', sz)
    if sz > CAP:
        print('SKIP over cap')
        lines.append('- SKIP over cap %s %s' % (rel, sz))
        continue
    out = dest_root / src.stem
    if out.exists():
        print('exists, skip', out)
        lines.append('- EXISTS ' + str(out))
        continue
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src) as z:
        z.extractall(out)
        names = z.namelist()[:30]
    print('extracted', len(list(out.rglob('*'))), 'into', out)
    lines.append('- OK %s -> %s first=%s' % (rel, out, names[:8]))
receipt = dest_root / 'EXTRACT.RECEIPT.md'
receipt.write_text('\n'.join(lines))
print('WROTE', receipt)
print('\n'.join(lines))
